In [3]:
import pandas as pd
import numpy as np
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

In [4]:
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))

DATA_PATH = os.path.join(
    BASE_DIR,
    "datasets",
    "raw",
    "stroke",
    "healthcare-dataset-stroke-data.csv"
)

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully!")
print("Original shape:", df.shape)

Dataset loaded successfully!
Original shape: (5110, 12)


In [5]:
df = df.drop(columns=["id"])

print("Shape after removing ID:", df.shape)

Shape after removing ID: (5110, 11)


In [6]:
X = df.drop(columns=["stroke"])
y = df["stroke"]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

Feature shape: (5110, 10)
Target shape: (5110,)


In [7]:
categorical_columns = [
    "gender",
    "ever_married",
    "work_type",
    "Residence_type",
    "smoking_status"
]

numerical_columns = [
    "age",
    "hypertension",
    "heart_disease",
    "avg_glucose_level",
    "bmi"
]

print("Categorical columns:")
print(categorical_columns)

print("\nNumerical columns:")
print(numerical_columns)

Categorical columns:
['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']

Numerical columns:
['age', 'hypertension', 'heart_disease', 'avg_glucose_level', 'bmi']


In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts())

print("\nTesting target distribution:")
print(y_test.value_counts())

Training set: (4088, 10)
Testing set: (1022, 10)

Training target distribution:
stroke
0    3889
1     199
Name: count, dtype: int64

Testing target distribution:
stroke
0    972
1     50
Name: count, dtype: int64


In [9]:
numeric_imputer = SimpleImputer(strategy="median")

X_train[numerical_columns] = numeric_imputer.fit_transform(
    X_train[numerical_columns]
)

X_test[numerical_columns] = numeric_imputer.transform(
    X_test[numerical_columns]
)

print("Missing values after numeric imputation:")
print(X_train[numerical_columns].isnull().sum())

Missing values after numeric imputation:
age                  0
hypertension         0
heart_disease        0
avg_glucose_level    0
bmi                  0
dtype: int64


In [10]:
X_train = pd.get_dummies(
    X_train,
    columns=categorical_columns,
    drop_first=False
)

X_test = pd.get_dummies(
    X_test,
    columns=categorical_columns,
    drop_first=False
)

X_train, X_test = X_train.align(
    X_test,
    join="left",
    axis=1,
    fill_value=0
)

print("Training shape after encoding:", X_train.shape)
print("Testing shape after encoding:", X_test.shape)

Training shape after encoding: (4088, 21)
Testing shape after encoding: (1022, 21)


In [11]:
X_train = X_train.astype(float)
X_test = X_test.astype(float)

print("Data types after encoding:")
print(X_train.dtypes.value_counts())

Data types after encoding:
float64    21
Name: count, dtype: int64


In [12]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaled training shape:", X_train_scaled.shape)
print("Scaled testing shape:", X_test_scaled.shape)

Scaled training shape: (4088, 21)
Scaled testing shape: (1022, 21)


In [13]:
feature_names = X_train.columns.tolist()

X_train_scaled = pd.DataFrame(
    X_train_scaled,
    columns=feature_names,
    index=X_train.index
)

X_test_scaled = pd.DataFrame(
    X_test_scaled,
    columns=feature_names,
    index=X_test.index
)

print("Final feature count:", len(feature_names))
print("\nFeature names:")
print(feature_names)

Final feature count: 21

Feature names:
['age', 'hypertension', 'heart_disease', 'avg_glucose_level', 'bmi', 'gender_Female', 'gender_Male', 'gender_Other', 'ever_married_No', 'ever_married_Yes', 'work_type_Govt_job', 'work_type_Never_worked', 'work_type_Private', 'work_type_Self-employed', 'work_type_children', 'Residence_type_Rural', 'Residence_type_Urban', 'smoking_status_Unknown', 'smoking_status_formerly smoked', 'smoking_status_never smoked', 'smoking_status_smokes']


In [14]:
print("Missing values in training data:")
print(X_train_scaled.isnull().sum().sum())

print("\nMissing values in testing data:")
print(X_test_scaled.isnull().sum().sum())

Missing values in training data:
0

Missing values in testing data:
0


In [15]:
print("Training target distribution:")
print(y_train.value_counts())

print("\nTraining target percentage:")
print(y_train.value_counts(normalize=True) * 100)

print("\nTesting target distribution:")
print(y_test.value_counts())

print("\nTesting target percentage:")
print(y_test.value_counts(normalize=True) * 100)

Training target distribution:
stroke
0    3889
1     199
Name: count, dtype: int64

Training target percentage:
stroke
0    95.132094
1     4.867906
Name: proportion, dtype: float64

Testing target distribution:
stroke
0    972
1     50
Name: count, dtype: int64

Testing target percentage:
stroke
0    95.107632
1     4.892368
Name: proportion, dtype: float64


In [16]:
PROCESSED_DIR = os.path.join(
    BASE_DIR,
    "datasets",
    "processed",
    "stroke"
)

os.makedirs(PROCESSED_DIR, exist_ok=True)

In [17]:
X_train_scaled.to_csv(
    os.path.join(PROCESSED_DIR, "x_train.csv"),
    index=False
)

X_test_scaled.to_csv(
    os.path.join(PROCESSED_DIR, "x_test.csv"),
    index=False
)

y_train.to_csv(
    os.path.join(PROCESSED_DIR, "y_train.csv"),
    index=False
)

y_test.to_csv(
    os.path.join(PROCESSED_DIR, "y_test.csv"),
    index=False
)

print("Processed datasets saved successfully!")

Processed datasets saved successfully!


In [18]:
import joblib

MODEL_DIR = os.path.join(
    BASE_DIR,
    "models",
    "stroke"
)

os.makedirs(MODEL_DIR, exist_ok=True)

joblib.dump(
    numeric_imputer,
    os.path.join(MODEL_DIR, "imputer.pkl")
)

joblib.dump(
    scaler,
    os.path.join(MODEL_DIR, "scaler.pkl")
)

joblib.dump(
    feature_names,
    os.path.join(MODEL_DIR, "feature_names.pkl")
)

print("Preprocessing objects saved successfully!")

Preprocessing objects saved successfully!


In [19]:
print("=" * 50)
print("STROKE PREPROCESSING SUMMARY")
print("=" * 50)

print("Original dataset:", df.shape)
print("Training features:", X_train_scaled.shape)
print("Testing features:", X_test_scaled.shape)

print("\nTraining samples:", len(y_train))
print("Testing samples:", len(y_test))

print("\nTraining stroke distribution:")
print(y_train.value_counts())

print("\nTesting stroke distribution:")
print(y_test.value_counts())

print("\nMissing values:")
print("Training:", X_train_scaled.isnull().sum().sum())
print("Testing:", X_test_scaled.isnull().sum().sum())

print("\nPreprocessing completed successfully!")

STROKE PREPROCESSING SUMMARY
Original dataset: (5110, 11)
Training features: (4088, 21)
Testing features: (1022, 21)

Training samples: 4088
Testing samples: 1022

Training stroke distribution:
stroke
0    3889
1     199
Name: count, dtype: int64

Testing stroke distribution:
stroke
0    972
1     50
Name: count, dtype: int64

Missing values:
Training: 0
Testing: 0

Preprocessing completed successfully!
